# Example usage of careful-scada
from data loading to building models.

## Data Loading & Preprocessing

In [ ]:
import sys, os
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go

sys.path.append(os.path.abspath(".."))
# add src folder to path
sys.path.append(os.path.abspath("../care_scada"))
print(sys.path)

from care_scada.data_loader import DataLoader
from care_scada.preprocessing import Preprocessor
from care_scada.plotter import plot_sensor_timeseries
from care_scada.data_factory import DatasetFactory
from care_scada.model_factory import ModelFactory

### Dataloading & Windfarm selection
The CARE_To_Compare dataset contains three windfarms with each containing multiple datasets itself. See the README.md from the 
authors for CARE_To_Compare for more details.
Lets load on of the farms:

Note how the loader prints the number of currently supported features. Also the number of loaded datesets from the farm is printed.

In [ ]:
farm_name = 'B'  # or 'B', 'C'
loader = DataLoader(farm_name=farm_name, data_path="dat/Care_To_Compare")  # relative to the repo root  # or 'B', 'C'
df = loader.load()

### Preprocessing & Dataset selection
The loaded data is now preprocessed according to the specifications and made ready for modelling.
Note how the Preprocessor offers different strategies for e.g. the handling of nan values. Also
it is up to you, if you want to drop oultiers, or only flag them as such.

For a faster experience, set an dataset id (ds_id) and only preprocess that specific dataset from the farm.

In [ ]:
ds_id = 77
pre = Preprocessor(
    farm_name=farm_name,
    dataset_id=ds_id,
    missing_values="drop",
    outlier_handling="drop",
)
df_train_clean, df_test_clean = pre.process(df)

Great, lets have a look at one of the datasets loaded. 

In [ ]:
plot_sensor_timeseries(
    df=df_train_clean, 
    dataset_id='77', 
    sensor_cols=pre.numeric_cols,
    df_test=df_test_clean,
    #show_power=True,
    show_outliers=True) 

# Modelling
Now the data is ready for modelling. Lets pass it in the DatasetFactory, that creates dataset_id level
dataframes for training.

In [ ]:
datasets = DatasetFactory.create(
    farm_name='B',
    train_df=df_train_clean,
    test_df=df_test_clean
)
# get the first dataset
ds = datasets[0]
print(ds.farm_name, ds.dataset_id, ds.train_df.shape, ds.test_df.shape)

Now the actual modelling starts: 
In this example, the target_col defines the target of six
different regression models, that are created using the ModelFactory. 
This setup allows to train models on all datasets of the the farm. 

In [ ]:
results = []
trained_models = {}
target_col = 'main_bearing_temp2_c_avg'

for ds in datasets:
    models = []
    for model_name in ['naive_forecaster', 'constant_forecaster', 'random_forest', 'lstm', 'pcrnn', 'kalman']:
        model = ModelFactory.create(
            model_name=model_name)
        model.fit(ds.train_df, target_col=target_col)
        predictions = model.predict(ds.test_df, target_col=target_col)
        rmse = root_mean_squared_error(ds.test_df[target_col], predictions[target_col])

        results.append({
            'farm_name': ds.farm_name,
            'dataset_id': ds.dataset_id,
            'model_name': model_name,
            'rmse': rmse
        })
        # Store model for later access
        key = (ds.farm_name, ds.dataset_id, model_name)
        trained_models[key] = model

Let's aggregate the resulting RMSE for the six implemented models and visualize the regression.

In [ ]:
# barplot, average RMSE per model
results_df = pd.DataFrame(results)
avg_rmse = results_df.groupby('model_name')['rmse'].mean()
avg_rmse.plot(kind='bar')
plt.ylabel('Average RMSE')
plt.title('Average RMSE per Model')
plt.show()

In [ ]:
ds = datasets[0]  # Visualize for the first dataset


fig = go.Figure()

models_list = ['naive_forecaster', 'constant_forecaster', 'random_forest', 'lstm', 'pcrnn', 'kalman']

# Traindata
fig.add_trace(go.Scatter(x=ds.train_df.index, y=ds.train_df[target_col],
                            mode='lines', name='Train Actual', 
                            line=dict(color='black', width=1),
                            legendgroup='actual'))
# Testdata
fig.add_trace(go.Scatter(x=ds.test_df.index, y=ds.test_df[target_col],
                            mode='lines', name='Test Actual', 
                            line=dict(color='gray', width=1),
                            legendgroup='actual'))

# Predictions
for model_name in models_list:
    key = (ds.farm_name, ds.dataset_id, model_name)
    model = trained_models.get(key)
    
    if model:
        train_preds = model.predict(ds.train_df, target_col=target_col)
        test_preds = model.predict(ds.test_df, target_col=target_col)

        # Train Prediction Trace
        fig.add_trace(go.Scatter(x=ds.train_df.index, y=train_preds[target_col],
                                    mode='lines', name=f'{model_name} (Train)',
                                    line=dict(dash='dash'),
                                    visible=False)) 
        # Test Prediction Trace
        fig.add_trace(go.Scatter(x=ds.test_df.index, y=test_preds[target_col],
                                    mode='lines', name=f'{model_name} (Test)',
                                    line=dict(dash='dash'),
                                    visible=False))

# Add Dropdown Buttons for Model Selection
buttons = []
start_idx_predictions = 2 # Index where predictions start (after the 2 Actual Traces)
traces_per_model = 2 # Train + Test Prediction per model

# Initialization: Make the first model visible
if len(fig.data) > start_idx_predictions:
    fig.data[start_idx_predictions].visible = True
    fig.data[start_idx_predictions+1].visible = True

for i, model_name in enumerate(models_list):
    # Create visibility mask: 
    # [True, True] (Actuals) + [False...] (all preds)
    visibility = [True, True] + [False] * (len(models_list) * traces_per_model)
    
    # Activate visibility for current model
    curr_idx = start_idx_predictions + (i * traces_per_model)
    # Ensure we don't go out-of-bounds (in case models are missing)
    if curr_idx + 1 < len(visibility):
        visibility[curr_idx] = True
        visibility[curr_idx + 1] = True
    
    button = dict(
        label=model_name,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Actual vs Predicted: {model_name}"}]
    )
    buttons.append(button)

# Layout Update with Menu
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=1.15, 
        xanchor="left",
        y=1.15,
        yanchor="top"
    )],
    title=f"Actual vs Predicted: {models_list[0]}",
    xaxis_title="Time",
    yaxis_title=target_col,
    template="plotly_white"
)

The physics-constrained RNN (`pcrnn`) is built on a lumped thermal network: the bearing temperature
exchanges heat with neighbouring measured temperatures and is heated by operating signals, all
with learned coefficients. The plot shows the network with the learned conductances, time constants and
temperature rises at full load.

### Defining a heat network
The `pcrnn` and `kalman` models and the detectors share one network description, the
`ThermalNetworkSpec`:

* `target`: the predicted temperature,
* `neighbours`: measured temperatures that exchange heat with the target (term $k\,(T_j - T_\text{target})$),
* `sources`: measured non-temperature signals (loads) that heat a state with a learned positive gain,
  either linearly or quadratically: `{column: 'linear' | 'square'}`. Use
  `{column: (node, transform)}` or `{column: [(node, transform), ...]}` to heat other states,
* `latent`: unmeasured nodes with their own heat balance (e.g. the nacelle air),
* `measured_states`: measured temperatures with their own heat balance. `with_measured_states`
  couples them to the target and gives them the target's neighbours and sources,
* `edges`: conductances between any two nodes (to connect `latent` nodes and `measured_states`).

The Kalman model adds a few options of its own: `cooling` (source links with a *negative* gain, e.g.
wind cooling the bearing housing), `const_heat` / `temp_heat` (a constant heat $c$ and a term
$\gamma\,(T_\text{out} - 10)$ per measured state) and `training`:

* `"kalman"`: maximum likelihood of the Kalman filter (the sensors are assimilated while fitting),
* `"observer"`: free-run error of the target, the measured states are assimilated,
* `"free"`: pure simulation, no sensor is assimilated (`model.simulate(df)` returns all states).

When no custom network is given, the models default to the network configuration in `sensor_config.yaml`.
Override single entries by adding keyword arguments with the same name (e.g. `ModelFactory.create('kalman', measured_states=[...])`) or define a custom network.
If a measured variable does not exist in the dataset, it is dropped.

The spec only describes the physics. A model *predicts* the spec's `target`; a detector *monitors*
one or more temperatures of the network (`monitored`, see Fault detection).

Attention! While the Kalman model splits the target into a latent node and an observed noisy measurement node, the PCRNN has no observation model and thus treats the observation of the target as the exact target value.


#### Default thermal network (PCRNN)

In [ ]:
from care_scada.plotter import plot_thermal_network

plot_thermal_network(trained_models[(ds.farm_name, ds.dataset_id, 'pcrnn')]).show()

#### Three predefined main-bearing networks
Three demonstration networks are provided for the two main bearings of Wind Farm B. All of them model
**both** bearings as states (bearing 2 = target, bearing 1 = measured state), and both bearings exchange
heat with the outside air and the gearbox bearing. They were compared by a structure search over
the six healthy turbines (free-run simulation of both bearings, see
`examples/Kalman_results/results/02_network_search/thermal_network_search.txt`):

| | network | heat inputs per bearing | parameters | test RMSE (both bearings) |
|---|---|---|---|---|
| 1 | **simple** (recommended default) | wind cooling $-\beta\,v_\text{wind}$, friction $\beta\,n_\text{rotor}^2$ | 8 | 1.02 K |
| 2 | **medium** | as 1, plus active power $\beta\,P$ | 10 | 0.99 K |
| 3 | **baseline** (the former demo) | rotor speed, power, constant heat $c$, $\gamma\,(T_\text{out}-10)$; bearings coupled | 16 | 1.29 K |

Switch the network for the rest of the notebook with `NETWORK = 1 | 2 | 3`. The free-run simulation
(`training="free"`) is used for the fit: both bearings are predicted only from the outside, gearbox
and operating signals, so a fault of one bearing cannot leak into the prediction of the other.

The plot of the spec draws every load as a heat input; whether a load cools is a model option
(`cooling`) and shows up after the fit as a negative steady-state rise.

In [ ]:
from care_scada.modeling.thermal_network import ThermalNetworkSpec
from care_scada.plotter import plot_thermal_network

BEARING2, BEARING1 = 'main_bearing_temp2_c_avg', 'main_bearing_temp1_c_avg'
OUTSIDE, GEARBOX = 'outside_temp_c_avg', 'gearbox_bearing_temp1_c_avg'
ROTOR, WIND, POWER = 'rotor_speed_rpm_avg', 'wind_speed1_ms_avg', 'active_power_kWh'
both = lambda transform: [(BEARING1, transform), (BEARING2, transform)]  # source into both bearings
decoupled = [(BEARING1, GEARBOX), (BEARING1, OUTSIDE), (BEARING2, GEARBOX), (BEARING2, OUTSIDE)]

NETWORKS = {
    1: dict(name='simple',
            spec=ThermalNetworkSpec(target=BEARING2, measured_states=[BEARING1], edges=decoupled,
                                    sources={WIND: both('linear'), ROTOR: both('square')}),
            cooling=[(WIND, BEARING1), (WIND, BEARING2)], const_heat=False, temp_heat=False),
    2: dict(name='medium',
            spec=ThermalNetworkSpec(target=BEARING2, measured_states=[BEARING1], edges=decoupled,
                                    sources={WIND: both('linear'), ROTOR: both('square'), POWER: both('linear')}),
            cooling=[(WIND, BEARING1), (WIND, BEARING2)], const_heat=False, temp_heat=False),
    3: dict(name='baseline',
            spec=ThermalNetworkSpec(target=BEARING2, neighbours=[OUTSIDE, GEARBOX],
                                    sources={ROTOR: 'linear', POWER: 'linear'}).with_measured_states([BEARING1]),
            cooling=[], const_heat=True, temp_heat=True),
}

NETWORK = 1  # <- choose 1 (simple), 2 (medium) or 3 (baseline)
net = NETWORKS[NETWORK]
network = net['spec']
# Kalman-model options of the chosen network (the spec itself holds only the structure)
net_kwargs = {k: net[k] for k in ('cooling', 'const_heat', 'temp_heat')}
print(f"network {NETWORK}: {net['name']}")
plot_thermal_network(network).show()

Fit the chosen network as a free-run simulation on the training part of the dataset. The plot of the
fitted model shows the learned rates (lines), the steady-state rise at full load of every heat input
(arrows, negative = cooling) and the time constants of the bearings.

Note: the test part of dataset 77 contains the bearing-2 fault, so bearing 2 runs several K warmer than
simulated there; that offset is exactly what the detectors below look for. On a healthy dataset
(e.g. `ds_id = 83` at the top) the free-run error of both bearings is about 1 K.

In [ ]:
kalman = ModelFactory.create('kalman', network=network, training='free', **net_kwargs)
kalman.fit(ds.train_df, target_col)
plot_thermal_network(kalman).show()

# free-run simulation of both bearings on the test part
sim = kalman.simulate(ds.test_df)
for col in kalman.sensors_:
    ok = ds.test_df[col].notna()
    print(f"test RMSE {col}: {root_mean_squared_error(ds.test_df[col][ok], sim[col][ok]):.2f} K")

fig = go.Figure()
for col, color in zip(kalman.sensors_, ['#E4572E', '#4C78A8']):
    fig.add_trace(go.Scatter(x=ds.test_df.index, y=ds.test_df[col], mode='lines', name=f'{col} measured',
                             line=dict(color=color, width=1)))
    fig.add_trace(go.Scatter(x=sim.index, y=sim[col], mode='lines', name=f'{col} simulated',
                             line=dict(color=color, width=1, dash='dot')))
fig.update_layout(title=f"Dataset {ds.dataset_id}, test part: network {NETWORK} ({net['name']}) free run",
                  yaxis_title='°C', template='plotly_white', legend=dict(orientation='h', y=-0.15))
fig.show()

# Fault detection
A bearing fault produces extra heat in one bearing. A **detector** compares each **monitored**
temperature (here both main bearings) with what its model expects from the operating conditions,
condenses the mismatch into one value per day, the **indicator** $x_d$ (in K), and raises an alarm
when the indicator grows unusually. It works in three steps:

1. `fit`: fit the detector's model on healthy data,
2. `calibrate`: compute the indicators of healthy turbines and set the alarm thresholds,
3. `detect`: list the alarms (day, monitored temperature, kind) of one turbine.

### Three detectors (indicators)
All three use the chosen `network` and monitor its target and measured state (bearing 2 and bearing 1).

| `DetectorFactory.create(...)` | model | indicator $x_d$ of bearing $i$ on day $d$ |
|---|---|---|
| `"residual"` | one PCRNN per bearing: the network predicts $T_i$ from the boundary temperatures and loads; a small neural net adds a correction that sees all other sensors (also the other bearing) | daily mean of $T_i^\text{measured} - T_i^\text{predicted}$ |
| `"freerun"` | the network simulates both bearings from the boundary temperatures and loads only; no bearing sensor is fed in | daily mean of $T_i^\text{measured} - T_i^\text{simulated}$ |
| `"kalman"` | Kalman filter on the network with both bearing sensors fed in; an unknown constant heat input $c_i$ per bearing is estimated continuously (random walk) | the steady-state temperature rise that $c_i$ causes: the extra heat the filter needs to explain bearing $i$, in K |

`kalman` and `freerun` adapt the fitted physics to every turbine on its first 90 days
(`calibration_days`), and no detector raises alarms in the first 90 days (`burn_in_days`).

### Two alarm routines
An alarm routine turns the indicator into a statistic $s_d$ and alarms when $s_d > h$. The threshold $h$
is the largest value of $s_d$ on the healthy turbines (times `margin`). Both routines work with every detector.

* **`cusum`** (`alarm="cusum"`, default of `residual` and `kalman`): two cumulative sums per bearing,
  $s_d = \max(0,\ s_{d-1} + \text{inc}_d - k)$:
  * `rise`: $\text{inc}_d = x_d - x_{d-1}$, $k$ = 0.02 K: a sustained fast increase,
  * `level`: $\text{inc}_d = x_d - \operatorname{median}(x_{d-90}, \dots, x_{d-30})$, $k$ = 0.5 K: an offset from the
    previous months.

  Weak spot: a fault that grows over months is absorbed by the moving median.
* **`level, cross`** (`alarm="level", reference="cross", margin=1.15`, default of `freerun`):
  1. cross reference: $z_i = x_i - \alpha\, x_j$ ($j$ = the other bearing, $\alpha \in [0, 1]$ fitted on days
     30–90). This removes drift that both bearings share (seasons, model error), but keeps the
     extra heat of one bearing,
  2. $s_d$ = mean of $z_i$ over the last 14 days minus the median of $z_i$ on days 30–90 (a fixed
     baseline: the turbine's own healthy start),
  3. alarm only if $s_d > h$ on 7 consecutive days.

`detector.with_alarm(...)` returns a copy with another routine that shares the fitted model, so both
routines are compared on the same indicators below.

**Setup.** The models are fitted on healthy data (dataset 83, and dataset 86 before its bearing-2
drift). Datasets 83, 52, 21, 2, 74, 82 are healthy turbines and set the thresholds; 77, 53 and 27 have
main-bearing damage (77 and 53: bearing 2).
For the full evaluation (leave-one-turbine-out thresholds, injected temperature ramps, all three
networks) see `examples/Kalman_results/results/03_fault_detection/fault_detection.txt`.

**Choose the detectors** with `DETECTORS` below (default: `freerun` only). Approximate runtime of one
detector (fit + indicators of 9 turbines) on a 12-core CPU: `freerun` 1.5 min, `kalman` 2.5 min,
`residual` 1.5 min. The Kalman filters run step by step and are 3–5× *slower* on a GPU; only the
`residual` PCRNN profits from one (about 0.5 min).

In [ ]:
from care_scada.detector_factory import DetectorFactory

DETECTORS = ['freerun']  # <- any of 'residual', 'kalman', 'freerun'

healthy_ids, fault_ids = [83, 52, 21, 2, 74, 82], [77, 53, 27]  # fault_ids: main-bearing damage
ids = healthy_ids + fault_ids + [86]

def turbine_series(ds_id, raw, repo_root):
    # train and test part of one dataset as a single time-indexed series
    sys.path.insert(0, repo_root)
    from care_scada.preprocessing import Preprocessor
    pre = Preprocessor(farm_name='B', dataset_id=ds_id, missing_values="drop",
                       outlier_handling="drop", verbose=False)
    return pd.concat(pre.process(raw)).droplevel(0).sort_index()

# preprocess the datasets in parallel (sequentially if joblib is not installed)
raw = {d: df.xs(str(d), level='dataset_id', drop_level=False) for d in ids}
try:
    from joblib import Parallel, delayed
    out = Parallel(n_jobs=min(len(ids), os.cpu_count()))(
        delayed(turbine_series)(d, raw[d], os.path.abspath("..")) for d in ids)
except ImportError:
    out = [turbine_series(d, raw[d], os.path.abspath("..")) for d in ids]
series = dict(zip(ids, out))
fit_df = pd.concat([series[83], series[86][:"2022-09-30"]])

create = {
    'residual': lambda: DetectorFactory.create('residual', network=network, model_name='pcrnn',
                                               model_kwargs={'cooling': net['cooling']}),
    'kalman': lambda: DetectorFactory.create('kalman', network=network, model_kwargs=net_kwargs),
    'freerun': lambda: DetectorFactory.create('freerun', network=network, **net_kwargs),
}
detectors = {name: create[name]() for name in DETECTORS}
bearings = network.states[:2]  # bearing 2 (target), bearing 1
print("monitored:", bearings)
indicators = {}
for name, det in detectors.items():
    det.fit(fit_df)
    indicators[name] = {d: det.indicators(series[d]) for d in healthy_ids + fault_ids}
    print(f"{name}: fitted, indicators done")

The network of the first chosen detector, here `freerun`: both bearings are monitored (red). Their
sensors are used only as the initial value of each gap-free sequence; the alarm indicator is measured
minus simulated temperature. (For `residual`, the PCRNN network of bearing 2 is shown.)

In [ ]:
det = detectors[DETECTORS[0]]
plot_thermal_network(det.models[BEARING2] if DETECTORS[0] == 'residual' else det).show()

### Results
For every detector and alarm routine: the healthy turbines with any alarm (false alarms; dataset 83 is
part of the fit, so it is not an independent test) and the first alarm on the faulty bearing of 77 / 53
(any bearing for 27, whose damaged bearing is not labelled) in days **before** the start of the CARE
fault label (negative = after the start).

In [ ]:
events = pd.read_csv(os.path.join(loader.path, "..", "event_info.csv"), sep=";").set_index("event_id")
FAULTY = {77: BEARING2, 53: BEARING2, 27: None}  # damaged bearing (CARE: "Rotor Bearing 2"; 27 unspecified)
ROUTINES = {
    'cusum': dict(alarm='cusum', reference='none', margin=1.0),
    'level, cross': dict(alarm='level', reference='cross', margin=1.15),
}

rows, alarm_lists = [], {}
for name, det in detectors.items():
    for routine, kwargs in ROUTINES.items():
        d_r = det.with_alarm(**kwargs)
        d_r.calibrate([indicators[name][d] for d in healthy_ids])
        false = [d for d in healthy_ids if len(d_r.detect(indicators[name][d]))]
        row = {'detector': name, 'alarm routine': routine, 'false alarms (of 6)': len(false)}
        for d in fault_ids:
            alarms = d_r.detect(indicators[name][d])
            alarm_lists[(name, routine, d)] = alarms
            if FAULTY[d] is not None:
                alarms = alarms[alarms['monitored'] == FAULTY[d]]
            start = pd.Timestamp(events.loc[d, 'event_start']).normalize()
            row[f'lead {d} (days)'] = (start - alarms['day'].iloc[0]).days if len(alarms) else None
        rows.append(row)
pd.DataFrame(rows).set_index(['detector', 'alarm routine'])

Results of all three detectors with network 1 (`DETECTORS = ['residual', 'kalman', 'freerun']`):

| detector | alarm routine | false alarms (of 6) | lead 77 | lead 53 | lead 27 |
|---|---|---|---|---|---|
| residual | cusum | 0 | −27 | – | – |
| residual | level, cross | 0 | −26 | −16 | −40 |
| kalman | cusum | 0 | 80 | −7 | 73 |
| kalman | level, cross | 0 | 53 | −14 | −30 |
| freerun | cusum | 0 | 88 | −14 | −36 |
| **freerun** | **level, cross** | **0** | **83** | **2** | **93** |

The `freerun` detector with the `level, cross` routine flags the slow bearing-2 fault of 77 about
three months before the CARE label and 53 at its start (its bearing-2 asymmetry rises only about two
weeks before); the early alarm on 27 is on bearing 1 (27's damaged bearing is not labelled). The
`residual` detector alarms only after the labelled start: its neural correction also sees the other
bearing, so part of the fault is predicted away. With the moving CUSUM baseline, slow faults are
detected late or missed. Switch `NETWORK` above to compare the three networks.

Dataset 77 (bearing 2 heats up slowly over months before the labelled fault): the signals the
`level, cross` routine works on (indicator of each bearing minus $\alpha$ times the other's) for the chosen
detectors, with the threshold and the first alarm.

In [ ]:
from plotly.subplots import make_subplots

d = 77
event = events.loc[d]
colors = dict(zip(bearings, ['#E4572E', '#4C78A8']))
fig = make_subplots(rows=len(detectors), cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[f"{name}: cross-referenced indicator" for name in detectors])
for i, (name, det) in enumerate(detectors.items(), start=1):
    d_r = det.with_alarm(**ROUTINES['level, cross'])
    h = d_r.calibrate([indicators[name][k] for k in healthy_ids])['level']
    sig = d_r.signals(indicators[name][d])
    base = sig.iloc[30:90].median()
    for b in bearings:
        fig.add_trace(go.Scatter(x=sig.index, y=sig[b] - base[b], mode='lines', name=b, legendgroup=b,
                                 showlegend=i == 1, line=dict(color=colors[b], width=1)), row=i, col=1)
    fig.add_hline(y=h, line_dash='dash', line_color='black', row=i, col=1)
    alarms = d_r.detect(indicators[name][d])
    for b, a in alarms.groupby('monitored'):
        fig.add_vline(x=a['day'].iloc[0], line_color=colors[b], line_width=2, row=i, col=1)
    fig.add_vrect(x0=event['event_start'], x1=event['event_end'], fillcolor='gray', opacity=0.2,
                  line_width=0, row=i, col=1)
fig.update_layout(height=250 + 250 * len(detectors), template='plotly_white', legend=dict(orientation='h', y=-0.08),
                  title=f"Dataset {d}: deviation from the baseline (K), threshold (dashed), first alarm "
                        f"(vertical line) and CARE fault label (grey, from {event['event_start'][:10]})")
fig.show()